# DimRed API Demo

This notebook demonstrates the complete workflow for using the DimRed API:

1. Create a project
2. Create a dataset
3. Add data points to the dataset
4. Create a prompt
5. Create a metric
6. Run prompt tuning
7. Poll for tuning session completion

## Setup

First, import the DimRed API client and configure logging.

In [ ]:
import json
import logging
import os
import sys

# Add parent directory to path to import client
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')), '..'))

from client import DimRedAPIClient

# Configure logging to output to stdout for better visibility in notebooks
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    stream=sys.stdout,  # Output to stdout instead of stderr
    force=True  # Override any existing configuration
)
logger = logging.getLogger(__name__)

## Configuration

Set your API key and base URL here.

In [ ]:
# Configuration
import os
API_KEY = os.environ.get("DIMRED_API_KEY")
BASE_URL = "https://api.dimred.com"

# Path to example data file (relative to notebook location)
DATA_FILE_PATH = os.path.join(os.path.dirname(os.path.abspath('')), '..', 'data', 'example.json')

# Initialize client
client = DimRedAPIClient(API_KEY, BASE_URL)

## Step 1: Create Project

In [3]:
project_id = client.create_project(
    project_name="Jupyter Demo Project",
    project_description="Testing DimRed API with Jupyter notebook"
)

print(f"✓ Created project: {project_id}")

[2025-10-21 23:58:31] INFO - Creating project: Jupyter Demo Project
[2025-10-21 23:58:32] INFO - ✓ Project created: 97e05fe6-c66d-4089-9903-f62e03a46640
✓ Created project: 97e05fe6-c66d-4089-9903-f62e03a46640


## Step 2: Create Dataset

In [4]:
dataset_id = client.create_dataset(
    project_id=project_id,
    dataset_name="Financial Crime Detection Dataset"
)

print(f"✓ Created dataset: {dataset_id}")

[2025-10-21 23:58:32] INFO - Creating dataset: Financial Crime Detection Dataset for project 97e05fe6-c66d-4089-9903-f62e03a46640
[2025-10-21 23:58:33] INFO - ✓ Dataset created: ds-ac43eb59-a710-4054-a12e-80355cd28f5c
✓ Created dataset: ds-ac43eb59-a710-4054-a12e-80355cd28f5c


## Step 3: Add Datapoints

Load data from the example file and add it to the dataset.

In [5]:
# Load data from file
with open(DATA_FILE_PATH, 'r') as f:
    example_data = json.load(f)

print(f"Loaded {len(example_data)} datapoints from {DATA_FILE_PATH}")

# Convert to API format
datapoints = []
for item in example_data:
    datapoints.append({
        "input_data": json.dumps(item["input"]),
        "expected_output": json.dumps(item["expected"])
    })

# Add to dataset
count = client.add_datapoints(dataset_id, datapoints)
print(f"✓ Added {count} datapoints")

Loaded 10 datapoints from /Users/jonathanjohannemann/Downloads/example_data (8).json
[2025-10-21 23:58:33] INFO - Adding 10 datapoints to dataset ds-ac43eb59-a710-4054-a12e-80355cd28f5c
[2025-10-21 23:58:34] INFO - ✓ Added 10 datapoints
✓ Added 10 datapoints


## Step 4: Create Prompt

Create a prompt for financial crime perpetrator detection.

In [6]:
messages = [
    {
        "prompt_text": (
            "You are an expert financial crime analyst. Your task is to analyze news article "
            "snippets and determine whether the person mentioned is a perpetrator of financial crime.\n\n"
            "A person is a PERPETRATOR if:\n"
            "- They are explicitly charged, indicted, arrested, or accused of financial crimes\n"
            "- There is clear evidence of illegal activity (e.g., court documents, bank records)\n"
            "- They are directly involved in illegal financial transactions\n\n"
            "A person is NOT a perpetrator if:\n"
            "- They are law enforcement, prosecutors, or investigators\n"
            "- They are witnesses, victims, or observers\n"
            "- There is only speculation or suspicion without charges\n"
            "- They are community leaders or officials responding to crimes\n\n"
            "Respond with JSON containing:\n"
            "- is_perpetrator: true or false\n"
            "- reasoning: brief explanation of your decision"
        ),
        "prompt_message_type": "system"
    }
]

# Create output schema for structured JSON response
output_schema = {
    "type": "object",
    "properties": {
        "is_perpetrator": {
            "type": "boolean",
            "description": "Whether the person is a perpetrator of financial crime"
        },
        "reasoning": {
            "type": "string",
            "description": "Brief explanation of the decision"
        }
    },
    "required": ["is_perpetrator", "reasoning"]
}

prompt_id = client.create_prompt(
    project_id=project_id,
    messages=messages,
    name="Financial Crime Perpetrator Detection",
    output_schema=output_schema
)

print(f"✓ Created prompt: {prompt_id}")

[2025-10-21 23:58:34] INFO - Creating prompt for project 97e05fe6-c66d-4089-9903-f62e03a46640
[2025-10-21 23:58:35] INFO - ✓ Prompt created: d0cede93-fb2f-4975-b406-4bcf71a5bf08
✓ Created prompt: d0cede93-fb2f-4975-b406-4bcf71a5bf08


## Step 5: Create Metric

Create a metric to evaluate perpetrator classification accuracy.

In [7]:
metric_code = '''
import json

def metric_func(output, expected):
    """
    Check if the LLM correctly identified whether someone is a perpetrator.
    Returns 1.0 for correct classification, 0.0 for incorrect.
    """
    # Parse output and expected if they're strings
    if isinstance(output, str):
        try:
            output = json.loads(output)
        except json.JSONDecodeError:
            return 0.0

    if isinstance(expected, str):
        try:
            expected = json.loads(expected)
        except json.JSONDecodeError:
            return 0.0

    # Extract is_perpetrator field
    output_value = output.get("is_perpetrator")
    expected_value = expected.get("is_perpetrator")

    # Both must be present and match
    if output_value is None or expected_value is None:
        return 0.0

    # Return 1.0 if they match, 0.0 if they don't
    return 1.0 if output_value == expected_value else 0.0
'''

metric_id = client.create_metric(
    project_id=project_id,
    code=metric_code,
    metric_name="Perpetrator Classification Accuracy",
    metric_description="Measures whether the model correctly identifies perpetrators vs non-perpetrators"
)

print(f"✓ Created metric: {metric_id}")

[2025-10-21 23:58:35] INFO - Creating metric for project 97e05fe6-c66d-4089-9903-f62e03a46640
[2025-10-21 23:58:36] INFO - ✓ Metric created: 0897da43-f4d4-4ac8-a8b3-6528a9de55c7
✓ Created metric: 0897da43-f4d4-4ac8-a8b3-6528a9de55c7


## Step 6: Run Tuning

Start a prompt tuning session.

In [8]:
tuning_result = client.run_tuning(
    project_id=project_id,
    dataset_id=dataset_id,
    prompt_id=prompt_id,
    metric_id=metric_id,
    num_iterations=3,
    model_name="gpt-4.1-mini",
    provider="openai"
)

session_id = tuning_result["tuning_session_id"]
task_id = tuning_result["task_id"]

print(f"✓ Tuning started")
print(f"  Session ID: {session_id}")
print(f"  Task ID: {task_id}")

[2025-10-21 23:58:36] INFO - Starting tuning session for project 97e05fe6-c66d-4089-9903-f62e03a46640
[2025-10-21 23:58:36] INFO -   Dataset: ds-ac43eb59-a710-4054-a12e-80355cd28f5c
[2025-10-21 23:58:36] INFO -   Prompt: d0cede93-fb2f-4975-b406-4bcf71a5bf08
[2025-10-21 23:58:36] INFO -   Metric: 0897da43-f4d4-4ac8-a8b3-6528a9de55c7
[2025-10-21 23:58:36] INFO -   Iterations: 3
[2025-10-21 23:58:36] INFO -   Model: gpt-4.1-mini (openai)
[2025-10-21 23:58:37] INFO - ✓ Tuning started
[2025-10-21 23:58:37] INFO -   Session ID: 7891bc4e-ded9-4bcc-97e6-3472f6e20f2b
[2025-10-21 23:58:37] INFO -   Task ID: 7891bc4e-ded9-4bcc-97e6-3472f6e20f2b
✓ Tuning started
  Session ID: 7891bc4e-ded9-4bcc-97e6-3472f6e20f2b
  Task ID: 7891bc4e-ded9-4bcc-97e6-3472f6e20f2b


## Step 7: Wait for Completion

Poll the tuning session until it completes. 

**Note:** This can take 10-30 minutes. Enable DEBUG logging below to see polling progress every 15 seconds.

In [9]:
# Optional: Enable DEBUG logging to see polling progress every 15 seconds
# Uncomment the line below if you want verbose output:
logging.getLogger().setLevel(logging.DEBUG)

In [10]:
final_result = client.wait_for_tuning_completion(
    session_id=session_id,
    poll_interval=15,
    timeout=3600
)

print("\n=== Final Results ===")
print(f"Session ID: {final_result['session_id']}")
print(f"Status: {final_result['status']}")
print(f"Best Metric Value: {final_result.get('best_metric_value', 'N/A')}")
print(f"Iterations: {final_result.get('max_iterations', 'N/A')}")

if final_result.get("metrics"):
    print("\nMetrics:")
    print(json.dumps(final_result["metrics"], indent=2))

[2025-10-21 23:58:37] INFO - Polling tuning session 7891bc4e-ded9-4bcc-97e6-3472f6e20f2b every 15s
[2025-10-21 23:58:37] INFO - Timeout: 3600s
[2025-10-21 23:58:37] DEBUG - Poll #1: Fetching session status (elapsed: 0.0s)
[2025-10-21 23:58:37] DEBUG - GET https://api.dimred.com/api/v2/prompts/tune/7891bc4e-ded9-4bcc-97e6-3472f6e20f2b
[2025-10-21 23:58:37] DEBUG - Starting new HTTPS connection (1): api.dimred.com:443
[2025-10-21 23:58:37] DEBUG - https://api.dimred.com:443 "GET /api/v2/prompts/tune/7891bc4e-ded9-4bcc-97e6-3472f6e20f2b HTTP/1.1" 200 258
[2025-10-21 23:58:37] DEBUG - Response status: 200
[2025-10-21 23:58:37] INFO - Status: in_progress
[2025-10-21 23:58:37] DEBUG - Sleeping for 15s before next poll
[2025-10-21 23:58:52] DEBUG - Poll #2: Fetching session status (elapsed: 15.8s)
[2025-10-21 23:58:52] DEBUG - GET https://api.dimred.com/api/v2/prompts/tune/7891bc4e-ded9-4bcc-97e6-3472f6e20f2b
[2025-10-21 23:58:52] DEBUG - Starting new HTTPS connection (1): api.dimred.com:443


## Step 8: Fetch Best Prompt

Retrieve the best performing prompt from the tuning session.

In [11]:
# Fetch the best prompt from the tuning session
best_prompt = client.get_best_prompt(session_id)

print("\n=== Best Prompt ===")
print(f"Prompt ID: {best_prompt['prompt_id']}")
print(f"Prompt Name: {best_prompt.get('name', 'N/A')}")
print(f"\nMessages:")
for i, msg in enumerate(best_prompt.get('messages', []), 1):
    print(f"\n--- Message {i} ({msg.get('prompt_message_type', 'unknown')}) ---")
    print(msg.get('prompt_text', ''))

if best_prompt.get('output_schema'):
    print(f"\nOutput Schema:")
    print(json.dumps(best_prompt['output_schema'], indent=2))

[2025-10-22 00:03:40] INFO - Fetching best prompt for session 7891bc4e-ded9-4bcc-97e6-3472f6e20f2b
[2025-10-22 00:03:40] DEBUG - GET https://api.dimred.com/api/v2/prompts/tune/7891bc4e-ded9-4bcc-97e6-3472f6e20f2b
[2025-10-22 00:03:40] DEBUG - Starting new HTTPS connection (1): api.dimred.com:443
[2025-10-22 00:03:41] DEBUG - https://api.dimred.com:443 "GET /api/v2/prompts/tune/7891bc4e-ded9-4bcc-97e6-3472f6e20f2b HTTP/1.1" 200 259
[2025-10-22 00:03:41] DEBUG - Response status: 200
[2025-10-22 00:03:41] DEBUG - GET https://api.dimred.com/api/v2/prompts/d0cede93-fb2f-4975-b406-4bcf71a5bf08
[2025-10-22 00:03:41] DEBUG - Starting new HTTPS connection (1): api.dimred.com:443
[2025-10-22 00:03:42] DEBUG - https://api.dimred.com:443 "GET /api/v2/prompts/d0cede93-fb2f-4975-b406-4bcf71a5bf08 HTTP/1.1" 200 1454
[2025-10-22 00:03:42] DEBUG - Response status: 200
[2025-10-22 00:03:42] INFO - ✓ Retrieved best prompt: d0cede93-fb2f-4975-b406-4bcf71a5bf08

=== Best Prompt ===
Prompt ID: d0cede93-fb2f

## Summary

You've successfully completed the full DimRed API workflow:

- ✓ Created a project
- ✓ Created a dataset with datapoints
- ✓ Created a prompt for financial crime detection
- ✓ Created a custom metric
- ✓ Ran prompt tuning
- ✓ Retrieved final results

You can now explore the results in the DimRed web interface or continue working with the API programmatically.